In [37]:
import torch
import torch.nn.functional as F
import os


In [38]:
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np

class SingleBasinDataset(Dataset):
    def __init__(self, basin_id,
                 area_id, data_dir,
                 train_rate, val_rate,
                 seq_len, tar_len, 
                 eps=1e-5, mode='train'):
        """
        Args:
            basin_id (str): 盆地编号，like "01013500"
            area_id (str): 区域编号，like "01"
            data_dir(str): 根目录
            seq_len (int): 输入序列长度 (encoder or decoder length)
            tar_len (int): 预测长度
            mode (str): 'train', 'val', or 'test' 用于划分数据
        """
        self.seq_len = seq_len
        self.target_len = tar_len

        'dataset/processed_data/01/01013500.csv'
        file_path = ".." + data_dir + f"/{area_id}/{basin_id}.csv"

        # 读取数据 (这里假设数据已经归一化好了，如果没有，需要在 __getitem__ 前处理)
        df = pd.read_csv(file_path)
        
        data = df.iloc[:, 1:].values  # 转为 numpy 数组
        
        # 2. 划分训练/验证/测试集 (简单的按时间切分)
        n = len(data)
        train_end = int(n * train_rate)
        val_end = int(n * val_rate) + train_end
        train_x = data[:train_end, :-1]
        train_y = data[:train_end, -1:]
        
        
        self.x_mean = np.mean(train_x, axis=0, keepdims=True)
        self.x_std = np.std(train_x, axis=0, keepdims=True)
        self.y_mean = np.mean(train_y, axis=0, keepdims=True)
        self.y_std = np.std(train_y, axis=0, keepdims=True)
        

        if mode == 'train':
            self.x_data = data[:train_end, :-1]
            self.y_data = data[:train_end, -1:]
        elif mode == 'val':
            self.x_data = data[train_end:val_end, :-1]
            self.y_data = data[train_end:val_end, -1:]
        else:
            self.x_data = data[val_end:, :-1]
            self.y_data = data[val_end:, -1:]
        
        self.x_data = (self.x_data - self.x_mean) / (self.x_std + eps)
        self.y_data = (self.y_data - self.y_mean) / (self.y_std + eps)

        # 3. 计算合法的样本数量
        # 因为要做滑动窗口，最后一段不足 seq_len + target_len 的数据不能用
        self.length = len(self.x_data) - self.seq_len - self.target_len + 1

    def __len__(self):
        # 如果数据太短，返回0
        return max(0, self.length)

    def __getitem__(self, idx):
        # 4. 滑动窗口取数
        # X: 从 idx 开始，取 seq_len 长
        x = self.x_data[idx: idx + self.seq_len]
        y = self.y_data[idx: idx + self.seq_len]
        
        
        y_val = y[-self.target_len:].copy()
        y[-self.target_len:] = 0
        

        # 转为 Tensor
        return torch.FloatTensor(x), torch.FloatTensor(y), torch.FloatTensor(y_val)
    

def get_dataloader(config, basin_id, area_id, mode='train'):
    dataset = SingleBasinDataset(
        basin_id=basin_id,
        area_id=area_id,
        data_dir=config['data']['path'],
        seq_len=config['data']['seq_len'],
        tar_len=config['data']['tar_len'],
        train_rate=config['data']['split_rate']['train_rate'],
        val_rate=config['data']['split_rate']['valid_rate'],
        mode=mode
    )


    loader = DataLoader(
        dataset,
        batch_size=config['data']['batch_size'],
        shuffle=(mode == 'train'),  # 训练时打乱，验证测试时不打乱
        num_workers=config['data']['num_workers']
    )

    return loader

In [39]:
import yaml
config = yaml.load(open('../config/config.yaml', 'r'), Loader=yaml.FullLoader)

In [40]:
loader = get_dataloader(config, '01013500', '01', mode='train')

In [41]:
for x, y, z in loader:
    print(x.shape, y.shape, z.shape)

torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([512, 22, 5]) torch.Size([512, 22, 1]) torch.Size([512, 2, 1])
torch.Size([97, 22, 5]) torch.Size([97, 22, 1]) torch.Size([97, 2, 1])


In [27]:
import torch
import torch.nn as nn
class FeedForwardNet(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_model = config['d_model']
        d_hidden = config['d_hidden']
        dropout = config['dropout']
        self.net = nn.Sequential(
            nn.Linear(d_model, d_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_hidden, d_model),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.net(x)


class BlockEncoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_model = config['d_model']
        n_head = config['n_head']
        dropout = config['dropout']
        self.attn = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.ffn = FeedForwardNet(config)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)

    def forward(self, x):
        attn_out = self.attn(x, x, x, need_weights=False)[0]  # (B, d_model , C)
        x = x + self.dropout1(attn_out)
        x = self.ln1(x)
        ffn_out = self.ffn(x)
        x = x + self.dropout2(ffn_out)
        x = self.ln2(x)
        return x


class Encoder(nn.Module):
    def __init__(self, config):
        super().__init__()
        d_features = config['d_features']
        d_model = config['d_model']
        seq_len = config['seq_len']
        n_layers = config['n_layers']

        self.linear = nn.Linear(in_features=d_features, out_features=d_model)
        self.position_embedding_table = nn.Embedding(seq_len, d_model)
        self.blocks = nn.Sequential(*[BlockEncoder(config) for _ in range(n_layers)])

    def forward(self, tar):
        B, L, C = tar.shape
        tar = self.linear(tar)  # (B, L, d_model)
        # print(tar.shape)
        pos_ids = torch.arange(L, device=tar.device)
        pos_embd = self.position_embedding_table(pos_ids)
        # print(pos_embd.shape)
        tar = tar + pos_embd
        tar = self.blocks(tar)

        return tar

class BlockDecoder(nn.Module):

    def __init__(self, config):
        super().__init__()

        d_model = config['d_model']
        n_head = config['n_head']
        dropout = config['dropout']

        self.self_attn = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, n_head, dropout=dropout, batch_first=True)
        self.ln1 = nn.LayerNorm(d_model)
        self.ln2 = nn.LayerNorm(d_model)
        self.ln3 = nn.LayerNorm(d_model)
        self.ffn = FeedForwardNet(config)
        self.dropout1 = nn.Dropout(dropout)
        self.dropout2 = nn.Dropout(dropout)
        self.dropout3 = nn.Dropout(dropout)

    def forward(self, tar, output_ecd, reason_mask, nan_mask):
        self_attn_out = self.self_attn(tar, tar, tar,
                                       attn_mask=reason_mask,
                                       key_padding_mask=nan_mask,
                                       need_weights=False)[0]
        tar = tar + self.dropout1(self_attn_out)
        tar = self.ln1(tar)

        cross_attn_out = self.cross_attn(tar, output_ecd, output_ecd, need_weights=False)[0]
        tar = tar + self.dropout2(cross_attn_out)
        tar = self.ln2(tar)

        ffn_out = self.ffn(tar)
        tar = tar + self.dropout3(ffn_out)
        tar = self.ln3(tar)

        return tar


class Decoder(nn.Module):

    def __init__(self, config):
        super().__init__()
        d_model = config['d_model']
        n_layers = config['n_layers']
        seq_len = config['seq_len']

        self.linear_in = nn.Linear(in_features=1, out_features=d_model)
        self.position_embedding_table = nn.Embedding(seq_len, d_model)
        self.blocks = nn.ModuleList([BlockDecoder(config) for _ in range(n_layers)])
        self.linear_out = nn.Linear(in_features=d_model, out_features=1)
        self.register_buffer('reason_mask', nn.Transformer.generate_square_subsequent_mask(seq_len).isinf())

    def forward(self, tar, output_ecd):
        B, L, C = tar.shape
        nan_mask = torch.isnan(tar).squeeze(-1)
        tar = torch.nan_to_num(tar, nan=0.0)

        tar = self.linear_in(tar)
        # print(tar.shape)
        pos_embd = self.position_embedding_table(torch.arange(L).to(tar.device))
        # print(pos_embd.shape)
        tar = tar + pos_embd

        for block in self.blocks:
            tar = block(tar, output_ecd,
                        reason_mask=self.reason_mask,
                        nan_mask=nan_mask)
        tar = self.linear_out(tar)
        return tar


class RR_Former(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.encoder = Encoder(config)
        self.decoder = Decoder(config)
        self.tar_len = config['tar_len']

    def forward(self, input_ecd, input_dcd):
        output_ecd = self.encoder(input_ecd)
        output_dcd = self.decoder(tar=input_dcd, output_ecd=output_ecd)

        return output_dcd[:, -self.tar_len:]
    

In [34]:
class NSELoss(nn.Module):
    
    def __init__(self, epsilon=0.1):
        super().__init__()
        self.epsilon = epsilon
        
    def forward(self, y_pred, y_true):
        
        mask = ~torch.isnan(y_true)
        # 为了保持维度以便后续广播计算，我们暂时把 NaN 填为 0
        y_true_filled = torch.nan_to_num(y_true, nan=0.0)
        y_pred_filled = y_pred * mask.float()
        
        squared_error = torch.sum((y_pred_filled - y_true_filled) ** 2, dim=1)
        valid_counts = mask.sum(dim=1)
        valid_counts = torch.clamp(valid_counts, min=1.0)
        sample_means = torch.sum(y_true_filled, dim=1) / valid_counts
        deviations = (y_true_filled - sample_means.unsqueeze(1)) * mask.float()
        squared_deviations = torch.sum(deviations ** 2, dim=1)
        
        per_sample_loss = squared_error / (squared_deviations + self.epsilon)
        
        valid_samples_mask = (mask.sum(dim=1) > 0).float()
        total_loss = torch.sum(per_sample_loss * valid_samples_mask)
        num_valid_samples = torch.sum(valid_samples_mask)
        
        if num_valid_samples == 0:
            return torch.tensor(0.0, device=y_pred.device, requires_grad=True)
        
        
        return total_loss / num_valid_samples

In [35]:
loss = NSELoss()

In [36]:
loss(output, z)

tensor(15.6525, grad_fn=<DivBackward0>)